# Movie Marketing Asset Generator
**V1** (zero-shot) → **V2** (RAG) → **V3 (agentic loop)


In [ ]:
# Cell 0: Install dependencies
!pip -q install "transformers>=4.41" "accelerate>=0.30" bitsandbytes \
                 sentence-transformers faiss-cpu pandas numpy requests tqdm \
                 langchain langchain-community

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 12.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.8/23.8 MB 37.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.5/2.5 MB 74.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 64.9/64.9 kB 7.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 75.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 51.0/51.0 kB 6.1 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-colab 1.0.0 requires requests==2.32.4, but you have requests 2.33.1 which is incompatible.


In [ ]:
# Cell 1: Imports & credentials
import os
import json
import re
import numpy as np
import pandas as pd
import requests
import torch
import faiss
from pprint import pprint
from tqdm import tqdm
from google.colab import userdata

TMDB_BEARER  = userdata.get("TMDB_BEARER")
HF_TOKEN     = userdata.get("HF_TOKEN")
GITHUB_TOKEN = userdata.get("GITHUB_TOKEN")

print(" Imports done")

 Imports done


In [ ]:
# Cell 2: Clone repo
!git clone https://{GITHUB_TOKEN}@github.com/ryuichi-github/ryuichi-github-info290-2026s-group5.git

Cloning into 'ryuichi-github-info290-2026s-group5'...
remote: Enumerating objects: 49, done.
remote: Counting objects: 100% (49/49), done.
remote: Compressing objects: 100% (36/36), done.
remote: Total 49 (delta 14), reused 37 (delta 9), pack-reused 0 (from 0)
Receiving objects: 100% (49/49), 2.55 MiB | 6.83 MiB/s, done.
Resolving deltas: 100% (14/14), done.


In [ ]:
# Cell 3: Load dataset
REPO_ROOT = "ryuichi-github-info290-2026s-group5"
file_path = os.path.join(REPO_ROOT, "tmdb-fetch/tmdb_movies.csv")

if os.path.exists(file_path):
    movies_df = pd.read_csv(file_path)
    print(f" Loaded {len(movies_df)} movies")
    print(movies_df.columns.tolist())
else:
    raise FileNotFoundError(f"{file_path} not found. Check repo clone.")

 Loaded 4816 movies
['id', 'title', 'tagline', 'overview', 'release_date', 'genre_ids', 'genre_names', 'vote_average', 'vote_count', 'popularity', 'poster_path', 'backdrop_path', 'runtime', 'original_language', 'status', 'revenue', 'budget', 'belongs_to_collection']


In [ ]:
# Cell 4: TMDB API helper + genre mapping
def tmdb_get(url, params=None):
    headers = {
        "Authorization": f"Bearer {TMDB_BEARER}",
        "Content-Type": "application/json;charset=utf-8"
    }
    r = requests.get(url, headers=headers, params=params, timeout=30)
    r.raise_for_status()
    return r.json()

tmdb_get("https://api.themoviedb.org/3/authentication")

genre_data = tmdb_get("https://api.themoviedb.org/3/genre/movie/list", params={"language": "en-US"})
GENRE_ID_TO_NAME = {g["id"]: g["name"] for g in genre_data.get("genres", [])}
GENRE_NAME_TO_ID = {v.lower(): k for k, v in GENRE_ID_TO_NAME.items()}

print(f" Genres loaded: {len(GENRE_ID_TO_NAME)}")

 Genres loaded: 19


In [ ]:
# Cell 5: Build RAG corpus
from sentence_transformers import SentenceTransformer

corpus_df = movies_df[
    movies_df["tagline"].notna() & (movies_df["tagline"].str.strip() != "")
].copy().reset_index(drop=True)

print(f"Total movies: {len(movies_df)}")
print(f"Corpus (tagline あり): {len(corpus_df)} ({len(corpus_df)/len(movies_df)*100:.1f}%)")
print(f"Excluded (tagline なし): {len(movies_df) - len(corpus_df)}")

embed_model = SentenceTransformer("sentence-transformers/all-MiniLM-L6-v2")

corpus_texts = (
    corpus_df["title"] + " " + corpus_df["tagline"] + " " + corpus_df["overview"]
).tolist()

embeddings = embed_model.encode(
    corpus_texts,
    normalize_embeddings=True,
    show_progress_bar=True
).astype("float32")

dim = embeddings.shape[1]
faiss_index = faiss.IndexFlatIP(dim)
faiss_index.add(embeddings)

print(f" Embeddings shape: {embeddings.shape}")
print(f" FAISS index size: {faiss_index.ntotal}")

Total movies: 4816
Corpus (tagline あり): 4312 (89.5%)
Excluded (tagline なし): 504


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Batches:   0%|          | 0/135 [00:00<?, ?it/s]

 Embeddings shape: (4312, 384)
 FAISS index size: 4312


In [ ]:
# Cell 6: Load LLM (Mistral-7B-Instruct-v0.2, 4-bit quantized)
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig

MODEL_NAME = "mistralai/Mistral-7B-Instruct-v0.2"

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_use_double_quant=True,
    bnb_4bit_compute_dtype=torch.float16,
)

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME, token=HF_TOKEN, use_fast=True)
model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    token=HF_TOKEN,
    device_map="auto",
    quantization_config=bnb_config,
)

print(" Model loaded on:", model.device)

config.json:   0%|          | 0.00/596 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

tokenizer.model:   0%|          | 0.00/493k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/414 [00:00<?, ?B/s]

model.safetensors.index.json: 0.00B [00:00, ?B/s]

Fetching 3 files:   0%|          | 0/3 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/291 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/111 [00:00<?, ?B/s]

 Model loaded on: cuda:0


In [ ]:
# Cell 7: Core functions - retrieval, generation, JSON extraction

def _normalize_genre_filter(genre_filter):
    if genre_filter is None:
        return None
    if isinstance(genre_filter, (list, tuple)) and len(genre_filter) > 0:
        if isinstance(genre_filter[0], str):
            ids = [GENRE_NAME_TO_ID[name.lower()] for name in genre_filter
                   if name.lower() in GENRE_NAME_TO_ID]
            return ids if ids else []
        if isinstance(genre_filter[0], int):
            return list(genre_filter)
    raise ValueError("genre_filter must be None, list of genre names, or list of genre ids.")


def retrieve(query: str, k: int = 5, genre_filter=None) -> pd.DataFrame:
    q = embed_model.encode([query], normalize_embeddings=True).astype("float32")
    gf_ids = _normalize_genre_filter(genre_filter)

    if gf_ids is None:
        scores, idx = faiss_index.search(q, k)
        out = corpus_df.iloc[idx[0]].copy()
        out["score"] = scores[0]
        return out.reset_index(drop=True)

    gf_ids = set(gf_ids)
    mask = corpus_df["genre_ids"].apply(
        lambda gids: any(gid in gf_ids for gid in eval(gids))
    ).to_numpy()
    candidate_idx = np.where(mask)[0]

    if candidate_idx.size == 0:
        scores, idx = faiss_index.search(q, k)
        out = corpus_df.iloc[idx[0]].copy()
        out["score"] = scores[0]
        out["note"] = "No genre-filtered candidates; fell back to unfiltered."
        return out.reset_index(drop=True)

    cand_emb = embeddings[candidate_idx]
    scores = cand_emb @ q[0]
    topk_local = np.argsort(-scores)[:k]
    topk_idx = candidate_idx[topk_local]
    out = corpus_df.iloc[topk_idx].copy()
    out["score"] = scores[topk_local]
    return out.reset_index(drop=True)


def refs_to_text(df: pd.DataFrame, n: int = 5) -> str:
    lines = []
    for _, r in df.head(n).iterrows():
        if pd.notna(r["tagline"]) and r["tagline"].strip():
            ov = (r["overview"] or "")[:200].replace("\n", " ")
            lines.append(f'- {r["title"]}: tagline: "{r["tagline"]}" | overview: "{ov}"')
    return "\n".join(lines)


def generate_text(prompt: str, max_new_tokens: int = 320) -> str:
    inputs = tokenizer(prompt, return_tensors="pt").to(model.device)
    with torch.no_grad():
        out = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            do_sample=True,
            temperature=0.6,
            top_p=0.9,
            repetition_penalty=1.10,
            pad_token_id=tokenizer.eos_token_id,
            eos_token_id=tokenizer.eos_token_id,
        )
    return tokenizer.decode(out[0], skip_special_tokens=True)


REQUIRED_KEYS = {"overview", "tagline", "poster_art_direction"}

def extract_best_json(text: str) -> dict:
    decoder = json.JSONDecoder()
    objs = []
    i = 0
    while True:
        start = text.find("{", i)
        if start == -1:
            break
        try:
            obj, end = decoder.raw_decode(text[start:])
            objs.append(obj)
            i = start + end
        except json.JSONDecodeError:
            i = start + 1
    if not objs:
        raise ValueError("No JSON object found in model output.")

    def score(obj):
        if not isinstance(obj, dict):
            return -10
        if not REQUIRED_KEYS.issubset(set(obj.keys())):
            return -5
        s = 0
        tagline  = obj.get("tagline", "")
        poster   = obj.get("poster_art_direction", "")
        overview = obj.get("overview", "")
        if isinstance(tagline, str) and len(tagline.strip()) > 0:
            s += 2
            if len(tagline.split()) <= 14:
                s += 1
        if isinstance(poster, str) and len(poster.strip()) > 20:
            s += 2
        if isinstance(overview, str) and len(overview.strip()) > 40:
            s += 2
        return s

    return max(objs, key=score)


print(" Core functions ready")

 Core functions ready


In [ ]:
# Cell 8: Prompt builder + V1/V2 runner

def build_prompt(user_setting: str, refs: str) -> str:
    return f"""You are a movie marketing generator.

The user has described their movie idea below. Based on this, generate marketing assets.

OUTPUT: ONE JSON object with EXACTLY these keys:
- overview (<= 80 words, compelling promo synopsis)
- tagline (<= 12 words)
- poster_art_direction (<= 60 words)

JSON FORMAT:
{{"overview": "", "tagline": "", "poster_art_direction": ""}}

USER'S MOVIE IDEA:
{user_setting}

REFERENCES from similar successful movies (use as inspiration for tone and style; do NOT copy):
{refs}

CONSTRAINTS:
- Output ONLY the JSON object (no markdown, no backticks, no commentary).
- tagline must be original and specific to this movie, not generic.
- overview must be written as compelling promo copy, not a plot summary.
- End immediately after the final '}}'.

Now output the JSON:
"""


def run_v1_v2(user_setting: str, k: int = 5, genre_filter=None):
    # V1: no RAG
    p1 = build_prompt(user_setting, refs="(none)")
    t1 = generate_text(p1)
    j1 = extract_best_json(t1)

    # V2: RAG
    topk = retrieve(user_setting, k=k, genre_filter=genre_filter)
    refs = refs_to_text(topk, n=k)
    p2 = build_prompt(user_setting, refs=refs)
    t2 = generate_text(p2)
    j2 = extract_best_json(t2)

    return j1, j2, topk, refs


print(" Prompt builder and runner ready")

 Prompt builder and runner ready


In [ ]:
# Cell 9: Run V1 + V2

user_setting = (
    "A low-budget psychological thriller set in a rainy rural village, "
    "where the only clinic is cut off by flooding."
)

v1, v2, topk, refs = run_v1_v2(
    user_setting, k=5, genre_filter=["thriller", "horror", "mystery"]
)

print("=== Retrieved movies ===")
display(topk[["title", "genre_names", "tagline", "score"]])

print("\n=== References passed to model ===")
print(refs)

print("\n=== V1: Zero-Shot ===")
pprint(v1)

print("\n=== V2: RAG ===")
pprint(v2)

=== Retrieved movies ===


,title,genre_names,tagline,score
0,Take Shelter,"['Thriller', 'Drama', 'Horror']",Far away from the cruel world.,0.489664
1,Hard Rain,"['Thriller', 'Crime', 'Action']",A simple plan. An instant fortune. Just add wa...,0.457851
2,Cure,"['Crime', 'Thriller', 'Horror', 'Mystery']",Madness. Terror. Murder.,0.427415
3,Regression,"['Horror', 'Mystery', 'Thriller']",Fear always finds its victim,0.415955
4,Delirium,"['Horror', 'Thriller']",It's all in your head,0.386188



=== References passed to model ===
- Take Shelter: tagline: "Far away from the cruel world." | overview: "Plagued by a series of apocalyptic visions, a young husband and father questions whether to shelter his family from a coming storm, or from himself."
- Hard Rain: tagline: "A simple plan. An instant fortune. Just add water." | overview: "An armored car driver tries to elude a gang of thieves while a flood ravages the countryside."
- Cure: tagline: "Madness. Terror. Murder." | overview: "A detective starts spiraling out of control when a wave of gruesome murders with seemingly similar bizarre circumstances is sweeping Tokyo."
- Regression: tagline: "Fear always finds its victim" | overview: "Minnesota, 1990. Detective Bruce Kenner investigates the case of young Angela, who accuses her father, John Gray, of an unspeakable crime. When John unexpectedly and without recollection admits guilt,"
- Delirium: tagline: "It's all in your head" | overview: "A man recently released from a ment

---
## V3: Agentic Loop

2. Judge scores it on 5 dimensions
3. If scores pass threshold → stop. If not → inject feedback and retry


In [ ]:
def make_gen_prompt(user_setting, refs, feedback=""):
    fb_block = f"PREVIOUS FEEDBACK:\n{feedback}" if feedback else ""
    return f"""You are a movie marketing generator.

    # On the first iteration feedback is empty so fb_block just disappears
    # from the prompt. On later iterations it carries in the judge's critique
    # from the previous round — that's what drives the refinement.

The user has described their movie idea below. Based on this, generate marketing assets.

OUTPUT: ONE JSON object with EXACTLY these keys:
- overview (<= 80 words, compelling promo synopsis)
- tagline (<= 12 words)
- poster_art_direction (<= 60 words)

# Showing the model an empty template so it knows exactly
# what structure we expect back — reduces malformed outputs.

JSON FORMAT:
{{"overview": "", "tagline": "", "poster_art_direction": ""}}

USER'S MOVIE IDEA:
{user_setting}

# These are the top-k similar movies pulled from our FAISS index.
# The model uses them for tone/style inspiration, not plot copying.

REFERENCES:
{refs}

# This section is blank on iteration 1, filled with judge feedback on 2+.

{fb_block}

CONSTRAINTS:
- Output ONLY the JSON object.
- tagline must be specific to this movie.
- overview must be promo copy, not a plot summary.
- End immediately after the final '}}'.

# Telling it to stop right after the closing brace cuts down on
# the model rambling after the JSON, which breaks our parser.

Now output the JSON:
"""


def make_judge_prompt(user_setting, assets):
    return f"""You are a film marketing evaluator.
Score the marketing assets below from 1-5 on each dimension.

 # This flips Mistral into evaluator mode instead of generator mode.
 # We pass the original movie idea so the judge can check whether
 # the generated assets actually match what was asked for.

MOVIE IDEA:
{user_setting}

ASSETS:
{assets}

DIMENSIONS:
1. narrative_fidelity - does it stick to the premise
2. genre_alignment - does it match the genre/tone
3. visual_specificity - is the poster description specific
4. creative_specificity - is the tagline original
5. output_format_validity - is it valid JSON with all keys

# These 5 dimensions map directly to the evaluation criteria in our milestone report.
# The judge scores each one so we know exactly where the output is failing,
# not just that it faile

OUTPUT this JSON with integer scores and one feedback sentence:
{{"narrative_fidelity": 0, "genre_alignment": 0, "visual_specificity": 0, "creative_specificity": 0, "output_format_validity": 0, "feedback": "what needs fixing"}}

Only output the JSON, nothing else.
"""

In [ ]:
# passing feedback from the judge back into the generator each round
# until the output is good enough or we hit the max iteration cap

def run_v3_student(user_setting, k=5, genre_filter=None):

    # Retrieve the top-k similar movies from our FAISS index once upfront
    # We reuse the same refs every iteration — only the feedback changes,
    topk = retrieve(user_setting, k=k, genre_filter=genre_filter)
    refs = refs_to_text(topk, n=k)

    # feedback starts empty on iteration 1...the generator gets no critique
    # until the judge has actually seen an output to score
    feedback = ""

    # history lets us trace what happened each round, which is useful for
    # we can show how scores improved over iterations
    history = []

    max_iter = 3

    for i in range(max_iter):
        print(f"\niteration {i+1}")
        gen_prompt = make_gen_prompt(user_setting, refs, feedback)
        result = generate_text(gen_prompt)

        # extract_best_json scans the raw model output for a valid JSON object
        try:
            assets = extract_best_json(result)
        except:
            # If we can't parse anything useful, log it and try the next iteration rather than crashing the whole loop
            print("couldnt parse output, trying again")
            history.append({"iter": i+1, "assets": None, "scores": None, "passed": False})
            continue

        print("tagline:", assets.get("tagline"))

        # Now we flip the model into evaluator mode using the judge prompt
        judge_prompt = make_judge_prompt(user_setting, json.dumps(assets))
        judge_raw = generate_text(judge_prompt)

        try:
            scores_out = extract_best_json(judge_raw)
        except:
            # If the judge output is broken we still save the assets we got
            print("judge output broken")
            history.append({"iter": i+1, "assets": assets, "scores": None, "passed": False})
            continue

        # Pull the 5 dimension scores out of the judge's JSON output.
        # Defaulting to 0 means a missing score counts as a fail,
        score_keys = ["narrative_fidelity", "genre_alignment", "visual_specificity",
                      "creative_specificity", "output_format_validity"]
        scores = {k: int(scores_out.get(k, 0)) for k in score_keys}

        # This one sentence from the judge is what gets injected into
        # it's the feedback loop mechanism
        feedback = scores_out.get("feedback", "")

        # All 5 dimensions need to hit 3+ to pass. If any single one fails we go around again. Using all() means there's no way to squeak through with one bad dimension and four good ones.
        passed = all(v >= 3 for v in scores.values())

        print("scores:", scores)
        print("feedback:", feedback)
        history.append({"iter": i+1, "assets": assets, "scores": scores, "passed": passed})

        if passed:
            # Early stop — no point running more iterations if we already
            # hit the quality threshold
            print("looks good, stopping early")
            break
        else:
            print("not good enough yet, trying again with feedback")

    # Walk backwards through history to find the last iteration that actually produced assets
    best = next((h for h in reversed(history) if h["assets"] is not None), None)
    return best["assets"] if best else None, history, topk, refs

In [ ]:
v3_student, hist_student, topk_s, refs_s = run_v3_student(
    user_setting, k=5, genre_filter=["thriller", "horror", "mystery"])

print("\nfinal output:")
pprint(v3_student)


iteration 1
tagline: Rain brings out the darkness
scores: {'narrative_fidelity': 0, 'genre_alignment': 0, 'visual_specificity': 0, 'creative_specificity': 0, 'output_format_validity': 0}
feedback: 
not good enough yet, trying again with feedback

iteration 2
tagline: Lifeblood amidst the rising waters
scores: {'narrative_fidelity': 0, 'genre_alignment': 0, 'visual_specificity': 0, 'creative_specificity': 0, 'output_format_validity': 0}
feedback: 
not good enough yet, trying again with feedback

iteration 3
tagline: Life and death collide in the heart of the flood.
scores: {'narrative_fidelity': 0, 'genre_alignment': 0, 'visual_specificity': 0, 'creative_specificity': 0, 'output_format_validity': 0}
feedback: 
not good enough yet, trying again with feedback

final output:
{'overview': 'In the isolated, rain-soaked rural village, the sole clinic '
             'becomes a battleground for life as the waters rise. As fear '
             'grips the residents, the doctor and nurse struggle 